In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.utils import *

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier

In [ ]:
# Read data files
x, y, x_test = load_processed_data()

# Test train split from sklearn model selection
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# LGBM Model configuration
model = LGBMClassifier(
    objective="binary",                 # Defines that it is a 2 state problem
    n_estimators=1000,                  # Number of trees that can be generated
    learning_rate=0.05,                 # Learning Rate
    num_leaves=31,                      # Size of each tree
    random_state=42                     # Ensures reproducability
)

# Training model
model.fit(
    x_train,                        # Training Features
    y_train,                        # Solutions to those features
    eval_set=[(x_val, y_val)],      # Evaluate performance for validation data
    eval_metric="auc"               # Specify ROC AUC as the performance metric
)

# Generating validation prediction
# Makes predictions and classifies them as probabilities over flat integers
# Also uses [:,1] as it returns two probabilitiy rows for no pit vs pit
val_prediction = model.predict_proba(x_val)[:,1]       

# Calculates auc score to evaluate performance of the model and prints
generate_auc(y_val, val_prediction)

# Generate predictions from test dataframe for submission
test_prediction = model.predict_proba(x_test)[:,1]

# Create submission 
generate_submission("../data/test.csv", test_prediction, "../submissions/lgbm_baseline_sub.csv")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 69905, number of negative: 281407
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004271 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1806
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 44
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392665
[LightGBM] [Info] Start training from score -1.392665
AUC: 0.9482


,id,PitNextLap
0,439140,0.004084
1,439141,0.001952
2,439142,0.003651
3,439143,0.143164
4,439144,0.822525
...,...,...
188160,627300,0.012314
188161,627301,0.636858
188162,627302,0.759464
188163,627303,0.828767


In [ ]:
check = pd.read_csv("../submissions/lgbm_baseline_sub.csv")

print(check.shape)
check.head()

<class 'NoneType'>
None
